# Preparation of the test files

In this Jupyter Notbook all necessary test files are loaded and prepared. This means the PointCloud datafiles as well as the necessary data to define the borders is imported and the laz-files cut so they only include the the areas on which the model will be tested on

In [18]:
import os
import pandas as pd
import geopandas as gpd
from geopandas import clip, overlay
import folium

### Function

In [21]:
def load_municipality_boundary(gmd_file: str, municipality_name: str) -> gpd.GeoDataFrame:
    """Load and return the boundary polygon for a given municipality."""
    gmd_borders = gpd.read_file(gmd_file, layer="UP_GEMEINDEN_F")
    # Clean names to remove any extra information in parentheses (e.g. "Wetzikon (ZH)" --> "Wetzikon")
    gmd_borders["GEMEINDENAME"] = (gmd_borders["GEMEINDENAME"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip())
    boundary = gmd_borders[gmd_borders["GEMEINDENAME"] == municipality_name].copy()
    return boundary

def load_and_filter_zoning(zoning_file: str, selected_names=None) -> gpd.GeoDataFrame:
    """Load Richtplan data and filter by selected OBJEKTART_NAME values."""
    if selected_names is None:
        selected_names = ["Zentrumsgebiet"]

    zoning = gpd.read_file(zoning_file, layer="KANT_RICHT_F")
    return zoning[zoning["OBJEKTART_NAME"].isin(selected_names)].copy()

def clip_zoning_to_municipality(zoning_gdf: gpd.GeoDataFrame, muni_boundary: gpd.GeoDataFrame, buffer_shrink=5) -> gpd.GeoDataFrame:
    """Clip zoning polygons to the buffered municipality boundary."""
    if zoning_gdf.crs != muni_boundary.crs:
        muni_boundary = muni_boundary.to_crs(zoning_gdf.crs)

    # Shrink boundary slightly to remove edge slivers
    muni_boundary["geometry"] = muni_boundary.geometry.buffer(-buffer_shrink)
    
    return clip(zoning_gdf, muni_boundary)

def show_on_map(gdf: gpd.GeoDataFrame, zoom=13):
    """Display GeoDataFrame on a Folium map."""
    gdf_wgs = gdf.to_crs(epsg=4326)
    centroid = gdf_wgs.geometry.union_all().centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=zoom)
    folium.GeoJson(gdf_wgs, name="Clipped Zoning").add_to(m)
    folium.LayerControl().add_to(m)
    return m

def process_municipality(
    zoning_file: str,
    gmd_file: str,
    municipality_name: str,
    output_dir: str = None,
    show_map=True
) -> gpd.GeoDataFrame:
    """
    Full processing pipeline:
    - Load and filter zoning data.
    - Load municipal boundary.
    - Clip zoning to boundary.
    - Save result as GPKG using municipality name.
    - Optionally preview on map.
    - Return GeoDataFrame for further processing.
    """
    zoning = load_and_filter_zoning(zoning_file)
    muni_boundary = load_municipality_boundary(gmd_file, municipality_name)
    clipped = clip_zoning_to_municipality(zoning, muni_boundary)

    # === Save clipped result ===
    if output_dir is None:
        output_dir = os.getcwd()  # current directory
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{municipality_name}_zoning.geojson")
    clipped.to_file(out_path, driver="GeoJSON")
    print(f"✅ Saved clipped zoning data to: {out_path}")

    if show_map:
        display(show_on_map(clipped))

    return clipped



#### Apply on top ten Municipalities with the most residents

In [22]:
output_directory = "C:/Users/flori/Desktop/pa-1_data/clipped_zonings"
richtplan_path = "C:/Users/flori/Desktop/pa-1_data/richtplan/dietikon/Kantonaler_Richtplan_-OGD/Kantonaler_Richtplan_-OGD.gpkg"
gmd_path = "C:/Users/flori/Desktop/pa-1_data/gemeindegrenzen/gemeindegrenzen.gpkg"
municipality = "Dietikon"

zoning_filtered = process_municipality(
    zoning_file=richtplan_path,
    gmd_file=gmd_path,
    municipality_name=municipality,
    output_dir=output_directory
)
display(zoning_filtered.head(3))
print(zoning_filtered.shape)

✅ Saved clipped zoning data to: C:/Users/flori/Desktop/pa-1_data/clipped_zonings\Dietikon_zoning.geojson


,OBJEKTART_ID,OBJEKTART_NAME,OBJEKTVERW,KTZH,KAPITEL_NAME,KAPITEL_ABKUERZUNG,UNTERKAPITEL_NAME,UNTERKAPITEL_NUMMER,NR_TABELLE,NAME_TABELLE,...,OBJEKTE_TEXT4,OBJEKTE_TEXT5,TITEL_TEXT1,TITEL_TEXT2,TITEL_TEXT3,TITEL_TEXT4,TITEL_TEXT5,SHAPE_AREA,SHAPE_LEN,geometry
24,102.0,Zentrumsgebiet,20309.0,1.0,SIEDLUNG,S,Zentrumsgebiet,2.3,203.1,Zentrumsgebiet,...,,,Gebiet,Zentrumsfunktion,Entwicklungsziele,,,1.841721e+06,7511.076258,"POLYGON ((2672213.648 1252782.3, 2672246.655 1..."


(1, 24)


### Clip LAZ files based on these polygons

In [5]:
import os
import json
import glob
import subprocess
from tqdm import tqdm
import geopandas as gpd
import laspy

def build_pdal_pipeline(input_laz, polygon_wkt, output_laz):
    return [
        { "type": "readers.las",   "filename": input_laz },
        { "type": "filters.crop",  "polygon": polygon_wkt },
        { "type": "writers.las",   "filename": output_laz }
    ]

def get_union_wkt(geojson_path):
    gdf = gpd.read_file(geojson_path)
    union_geom = gdf.geometry.union_all()
    return union_geom.wkt

def clip_laz_files(input_root, zoning_dir, municipality_name, output_root, overwrite=False):
    """
    - input_root:  e.g. .../pa-1_data/test
    - zoning_dir:  .../pa-1_data/clipped_zonings
    - municipality_name: "Zürich"
    - output_root: .../pa-1_data/clipped_files
    """
    # Build per-municipality subfolders
    input_dir  = os.path.join(input_root, municipality_name)
    output_dir = os.path.join(output_root, municipality_name)
    os.makedirs(output_dir, exist_ok=True)

    # Find the GeoJSON for this municipality
    geojson_file = os.path.join(zoning_dir, f"{municipality_name}_zoning.geojson")
    if not os.path.exists(geojson_file):
        raise FileNotFoundError(f"Zoning file not found: {geojson_file}")
    print(f"Using zoning file: {geojson_file}")

    # Inline WKT polygon
    polygon_wkt = get_union_wkt(geojson_file)

    # Gather tiles
    laz_files = glob.glob(os.path.join(input_dir, "*.las")) + \
                glob.glob(os.path.join(input_dir, "*.laz"))
    if not laz_files:
        print(f"No .las/.laz files in {input_dir}")
        return
    print(f"Found {len(laz_files)} tiles to process in {municipality_name}.")

    for laz_file in tqdm(laz_files, desc=f"Cropping {municipality_name}"):
        name     = os.path.splitext(os.path.basename(laz_file))[0]
        out_path = os.path.join(output_dir, f"{name}_cropped.laz")

        # Skip if existing and not overwrite
        if os.path.exists(out_path) and not overwrite:
            continue

        # Build & write pipeline JSON
        pipeline = build_pdal_pipeline(laz_file, polygon_wkt, out_path)
        pipeline_file = os.path.join(output_dir, f"{name}_pipeline.json")
        with open(pipeline_file, 'w') as f:
            json.dump(pipeline, f, indent=2)

        # Run PDAL
        try:
            subprocess.run(
                ["pdal", "pipeline", pipeline_file],
                check=True, capture_output=True, text=True
            )
        except subprocess.CalledProcessError as e:
            print(f"Error processing {name}:\n{e.stderr.strip()}")
            os.remove(pipeline_file)
            continue
        finally:
            # Clean up pipeline file
            if os.path.exists(pipeline_file):
                os.remove(pipeline_file)

        # Check for empty output and delete if zero points
        try:
            with laspy.open(out_path) as f:
                count = f.header.point_count
            if count == 0:
                os.remove(out_path)
                print(f"➡️ Removed empty crop: {name}_cropped.laz")
        except Exception as e:
            print(f"⚠️ Could not verify output {out_path}: {e}")


In [14]:
INPUT_ROOT        = r"C:/Users/flori/Desktop/pa-1_data/test"
ZONING_DIR        = r"C:/Users/flori/Desktop/pa-1_data/clipped_zonings"
MUNICIPALITY_NAME = "Dietikon"
OUTPUT_ROOT       = r"C:/Users/flori/Desktop/pa-1_data/clipped_files"

clip_laz_files(
    input_root=INPUT_ROOT,
    zoning_dir=ZONING_DIR,
    municipality_name=MUNICIPALITY_NAME,
    output_root=OUTPUT_ROOT,
    overwrite=False
)

Using zoning file: C:/Users/flori/Desktop/pa-1_data/clipped_zonings\Dietikon_zoning.geojson
Found 21 tiles to process in Dietikon.


Cropping Dietikon:   0%|          | 0/21 [00:00<?, ?it/s]

Cropping Dietikon:  10%|▉         | 2/21 [00:06<00:56,  3.00s/it]

➡️ Removed empty crop: 2671000_1252000_cropped.laz


Cropping Dietikon:  57%|█████▋    | 12/21 [01:00<00:49,  5.45s/it]

➡️ Removed empty crop: 2672500_1250000_cropped.laz


Cropping Dietikon:  86%|████████▌ | 18/21 [01:33<00:13,  4.49s/it]

➡️ Removed empty crop: 2673000_1250000_cropped.laz


Cropping Dietikon: 100%|██████████| 21/21 [01:44<00:00,  5.00s/it]


#### Test with few files and visualizatin with open3d

In [ ]:
import laspy
import numpy as np
import open3d as o3d
import glob
import os

def load_xyz_from_laz(path):
    with laspy.open(path) as f:
        pts = f.read().points
        return np.vstack((pts.x, pts.y, pts.z)).T

def make_pc(xyz, color):
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(xyz)
    pc.paint_uniform_color(color)
    return pc

# ─── SETTINGS ──────────────────────────────────────────────────────────────
orig_dir   = r"C:/Users/flori/Desktop/pa-1_data/test/Zürich"
crop_dir   = r"C:/Users/flori/Desktop/pa-1_data/clipped_files/Zürich"
orig_glob  = os.path.join(orig_dir, "*.laz")
crop_glob  = os.path.join(crop_dir, "*_cropped.laz")
max_points = 5_000_000
# ────────────────────────────────────────────────────────────────────────────

# 1) Collect file lists
orig_files = sorted(glob.glob(orig_glob))
crop_files = sorted(glob.glob(crop_glob))
print(f"Found {len(orig_files)} original tiles and {len(crop_files)} cropped tiles.")

# 2) Load & color
all_clouds = []
total_pts  = 0

for fp in orig_files:
    xyz = load_xyz_from_laz(fp)
    all_clouds.append(make_pc(xyz, color=[0.7,0.7,0.7]))
    total_pts += len(xyz)

for fp in crop_files:
    xyz = load_xyz_from_laz(fp)
    all_clouds.append(make_pc(xyz, color=[1.0,0.0,0.0]))
    total_pts += len(xyz)

print(f"Total points loaded: {total_pts:,}")

# 3) Downsample if too many points
if total_pts > max_points:
    ratio = max_points / total_pts
    print(f"Downsampling all clouds by ratio {ratio:.3f} for performance")
    all_clouds = [pc.random_down_sample(ratio) for pc in all_clouds]


# 4) Visualize
o3d.visualization.draw_geometries(
    all_clouds,
    window_name="Gray = original | Red = cropped",
    width=1024, height=768
)


Found 3 original tiles and 2 cropped tiles.
Total points loaded: 46,289,440
Downsampling all clouds by ratio 0.108 for performance
